# D402 — Practical Snowflake Tags, Comments, and Data Governance


This standalone lab uses synthetic customer data. It does not depend on the earlier roles notebook or modify its objects. You will create reusable tags, apply them to databases, schemas, tables, table columns, views, and view columns, write comments, inspect effective metadata, and update or remove assignments.

Basic tag creation and assignment are available to all Snowflake accounts. Automatic tag propagation and tag-based masking require Enterprise Edition or higher. [Snowflake tagging availability](https://docs.snowflake.com/en/user-guide/object-tagging/introduction).

The SQL was reviewed against official documentation, but was not executed against a live Snowflake account while preparing this notebook.

## 1. What a tag means in governance

**Data governance** defines responsibilities, permitted uses, protection rules, and lifecycle decisions. **Metadata** describes data: its meaning, source, owner, classification, or handling requirements.

A **tag** is a named metadata object associated with a value on another object. Think of three separate pieces:

| Piece | Example | Meaning |
|---|---|---|
| Tag definition | `BUSINESS_OWNER` | The reusable question: which team is accountable? |
| Allowed value | `CUSTOMER_OPERATIONS` | One permitted answer |
| Assignment | Customer table gets that value | The answer for this particular asset |

A Snowflake tag lives inside a database and schema, even when it is applied to a different database or schema. Its location is not the same as the location of the tagged object. [Introduction to object tagging](https://docs.snowflake.com/en/user-guide/object-tagging/introduction).

Our fictional retailer wants to discover customer datasets, find responsible teams, and distinguish contact columns from ordinary business fields. A machine-readable tag vocabulary makes those questions easier to answer consistently.

## 2. Tags, comments, and policies have different jobs

A **comment** is free-text documentation. A **policy** is an enforceable rule. **Personally Identifiable Information (PII)** can identify a person directly or through linkage.

| Mechanism | Example | What it provides |
|---|---|---|
| Tag | `SENSITIVITY = RESTRICTED` | Structured classification |
| Comment | “Customer contact email; synthetic in this lab” | Explanation for people |
| Role privilege | Permission to read a table | Object access |
| Masking policy | Return `[hidden]` for selected roles | Value protection |
| Row access policy | Show only approved regions | Record filtering |
| Retention workflow | Delete eligible records after review | Lifecycle implementation |

A PII tag does not hide a column. A comment saying “do not export” does not prevent exports. A retention tag does not schedule deletion by itself.

Use tags for controlled values that people or automation will filter on. Use comments to explain meaning, units, grain, limitations, and intended use. Connect policies and workflows separately.

## 3. Choose a small, meaningful vocabulary

The names below are organizational examples, not Snowflake system tags. We use ordinary single-value tags in this lesson.

| Tag | Allowed values | Intended use |
|---|---|---|
| `BUSINESS_OWNER` | `DATA_PLATFORM`, `CUSTOMER_OPERATIONS`, `ANALYTICS` | Accountable team |
| `SENSITIVITY` | `INTERNAL`, `CONFIDENTIAL`, `RESTRICTED` | Handling classification |
| `DATA_CATEGORY` | `CUSTOMER_ID`, `EMAIL`, `REGION`, `ORDER_COUNT` | Meaning of a column |
| `CONTAINS_PII` | `YES`, `NO` | Dataset-level personal-data signal |
| `RETENTION_CLASS` | `CUSTOMER_LIFECYCLE`, `REPORTING_REVIEW` | Reference to a reviewed lifecycle rule |

Avoid separate spellings such as `restricted`, `Restricted`, and `HighlyRestricted` when they mean the same thing. Define who approves vocabulary changes and what each value requires.

`CUSTOMER_ID` remains potentially personal because it can link to identity records. A missing tag does not establish that a column is safe. `CONTAINS_PII = NO` should require review, not be the automatic answer when discovery finds nothing.

Do not place real customer details, passwords, or secrets in tag values or comments. Metadata can be visible to users who discover objects.

## 4. Lab permissions and objects

The instructor needs permission to use `SYSADMIN`, the System Administrator role. This lab creates and owns all objects under that role. A real deployment can separate a central tag administrator from data owners.

Creating a tag requires `CREATE TAG` on its schema. Applying a tag can be authorized through global `APPLY TAG`, or through `APPLY` on the tag plus ownership of the target. Column assignments use ownership of the containing table or view. Owning the lab tags and targets gives this lab the necessary authority. [Tagging privileges](https://docs.snowflake.com/en/user-guide/object-tagging/work#summary-of-ddl-commands-operations-and-privileges).

| Object | Name | Purpose |
|---|---|---|
| Database | `D402_TAG_LAB` | Isolated lab |
| Schema | `GOVERNANCE` | Reusable tag definitions |
| Schema | `SALES` | Synthetic table and view |
| Warehouse | `D402_TAG_WH` | Small compute resource |

A **warehouse** provides compute and consumes credits while operating. This lab uses an extra-small warehouse with automatic suspension after 60 seconds of inactivity.

Check for existing names before setup. If these belong to other work, use a different prefix consistently. `CREATE` intentionally stops on collisions rather than replacing existing objects.

```sql
USE ROLE SYSADMIN;
SHOW DATABASES LIKE 'D402_TAG_LAB';
SHOW WAREHOUSES LIKE 'D402_TAG_WH';
```

## 5. Create the environment and initial descriptions

A **database** contains schemas. A **schema** groups related objects. We place tag definitions in `GOVERNANCE` and business objects in `SALES` to make their different purposes visible.

```sql
USE ROLE SYSADMIN;
USE SECONDARY ROLES NONE;

CREATE WAREHOUSE D402_TAG_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE;

CREATE DATABASE D402_TAG_LAB
  COMMENT = 'D402 standalone tag and documentation training lab';

CREATE SCHEMA D402_TAG_LAB.GOVERNANCE
  COMMENT = 'Controlled metadata definitions for the training lab';

CREATE SCHEMA D402_TAG_LAB.SALES
  COMMENT = 'Synthetic customer data and reporting views';

USE WAREHOUSE D402_TAG_WH;
```

The initial comments explain why these containers exist. They are not classifications or access grants. We will update and inspect comments later.

## 6. Create tags with descriptions and allowed values

`ALLOWED_VALUES` restricts which strings can be assigned. A tag's own `COMMENT` explains how people should use that tag. Creating the definition does not assign it to business objects. [CREATE TAG](https://docs.snowflake.com/en/sql-reference/sql/create-tag).

```sql
CREATE TAG D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER
  ALLOWED_VALUES 'DATA_PLATFORM', 'CUSTOMER_OPERATIONS', 'ANALYTICS'
  COMMENT = 'Accountable business team; not Snowflake object ownership';

CREATE TAG D402_TAG_LAB.GOVERNANCE.SENSITIVITY
  ALLOWED_VALUES 'INTERNAL', 'CONFIDENTIAL', 'RESTRICTED'
  COMMENT = 'Reviewed handling classification; does not mask data by itself';

CREATE TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY
  ALLOWED_VALUES 'CUSTOMER_ID', 'EMAIL', 'REGION', 'ORDER_COUNT'
  COMMENT = 'Business meaning of a column, independent of sensitivity';

CREATE TAG D402_TAG_LAB.GOVERNANCE.CONTAINS_PII
  ALLOWED_VALUES 'YES', 'NO'
  COMMENT = 'Reviewed dataset-level personal-information signal';

CREATE TAG D402_TAG_LAB.GOVERNANCE.RETENTION_CLASS
  ALLOWED_VALUES 'CUSTOMER_LIFECYCLE', 'REPORTING_REVIEW'
  COMMENT = 'Reference to an approved lifecycle rule; no automatic deletion';

SHOW TAGS IN SCHEMA D402_TAG_LAB.GOVERNANCE;

SELECT SYSTEM$GET_TAG_ALLOWED_VALUES(
  'D402_TAG_LAB.GOVERNANCE.SENSITIVITY'
) AS ALLOWED_SENSITIVITY_VALUES;
```

Expected: five tag definitions; sensitivity permits the three listed strings. [Allowed-value management](https://docs.snowflake.com/en/user-guide/object-tagging/work#set-a-list-of-allowed-tag-values).

## 7. Tag a database

A database-level assignment is useful for a broad owner or handling default. Child objects can inherit it; a more specific assignment can override its effective value.

```sql
ALTER DATABASE D402_TAG_LAB SET TAG
  D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER = 'DATA_PLATFORM',
  D402_TAG_LAB.GOVERNANCE.SENSITIVITY = 'INTERNAL';

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER',
  'D402_TAG_LAB', 'DATABASE'
) AS DATABASE_OWNER_LABEL;
```

Expected: `DATA_PLATFORM`. This is a business label. It does not transfer technical ownership of the database, which remains with `SYSADMIN`.

`SYSTEM$GET_TAG` returns one effective tag value for an object. Supply the tag name, target name, and object domain. An unassociated tag returns SQL `NULL`, meaning no value, subject to visibility and valid object references. [SYSTEM$GET_TAG](https://docs.snowflake.com/en/sql-reference/functions/system_get_tag).

## 8. Tag a schema

The sales team has a more specific owner and classification than the database-wide default.

```sql
ALTER SCHEMA D402_TAG_LAB.SALES SET TAG
  D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER = 'CUSTOMER_OPERATIONS',
  D402_TAG_LAB.GOVERNANCE.SENSITIVITY = 'CONFIDENTIAL';

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER',
  'D402_TAG_LAB.SALES', 'SCHEMA'
) AS SALES_OWNER;

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.SENSITIVITY',
  'D402_TAG_LAB.GOVERNANCE', 'SCHEMA'
) AS GOVERNANCE_SCHEMA_SENSITIVITY;
```

Expected: `CUSTOMER_OPERATIONS` for Sales and inherited `INTERNAL` for the Governance schema.

**Inheritance** follows supported containment relationships: database to schema to table or view to column. It does not mean every object that reads another object automatically copies its tags. [Tag inheritance](https://docs.snowflake.com/en/user-guide/object-tagging/inheritance).

## 9. Create and describe a customer table

A table's **grain** is what one row represents. A useful description includes grain, source, business meaning, units, and limitations.

```sql
CREATE TABLE D402_TAG_LAB.SALES.CUSTOMERS (
  CUSTOMER_ID INTEGER COMMENT 'Synthetic customer reference; one row per customer',
  EMAIL VARCHAR COMMENT 'Synthetic contact email; no real customer information',
  REGION VARCHAR COMMENT 'Sales region label, such as WEST or EAST',
  ORDER_COUNT INTEGER COMMENT 'Illustrative lifetime completed-order count'
)
COMMENT = 'Synthetic customer summary; grain is one row per customer';

INSERT INTO D402_TAG_LAB.SALES.CUSTOMERS
  (CUSTOMER_ID, EMAIL, REGION, ORDER_COUNT)
VALUES
  (1, 'learner1@example.invalid', 'WEST', 4),
  (2, 'learner2@example.invalid', 'EAST', 7);

DESCRIBE TABLE D402_TAG_LAB.SALES.CUSTOMERS;
```

Expected: four columns and their comments. `ORDER_COUNT` is a count, not a currency amount. Stating that distinction prevents a consumer from making an incorrect assumption.

Comments can be supplied during creation or maintained later with `COMMENT ON`. [COMMENT command](https://docs.snowflake.com/en/sql-reference/sql/comment).

## 10. Tag a table

Table tags describe the dataset as a whole. They do not replace field-by-field classification.

```sql
ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS SET TAG
  D402_TAG_LAB.GOVERNANCE.CONTAINS_PII = 'YES',
  D402_TAG_LAB.GOVERNANCE.RETENTION_CLASS = 'CUSTOMER_LIFECYCLE';

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.SENSITIVITY',
  'D402_TAG_LAB.SALES.CUSTOMERS', 'TABLE'
) AS CUSTOMER_TABLE_SENSITIVITY;

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.CONTAINS_PII',
  'D402_TAG_LAB.SALES.CUSTOMERS', 'TABLE'
) AS PERSONAL_DATA_SIGNAL;
```

Expected: inherited `CONFIDENTIAL` sensitivity and directly assigned `YES`.

We deliberately treat these synthetic fields as if they represented a customer dataset for classification practice. The tag is a teaching label, not a claim that these invented rows identify real people.

A table-level `CONTAINS_PII` tag can also appear through inheritance on its columns. That does not prove every individual column is an identifier. Use a dedicated column category for precise meaning.

## 11. Tag individual table columns

Column tags provide precise classification. The email receives a stricter sensitivity than its containing schema.

```sql
ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS MODIFY COLUMN CUSTOMER_ID
  SET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY = 'CUSTOMER_ID';

ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS MODIFY COLUMN EMAIL
  SET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY = 'EMAIL',
          D402_TAG_LAB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';

ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS MODIFY COLUMN REGION
  SET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY = 'REGION';

ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS MODIFY COLUMN ORDER_COUNT
  SET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY = 'ORDER_COUNT';

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.SENSITIVITY',
  'D402_TAG_LAB.SALES.CUSTOMERS.EMAIL', 'COLUMN'
) AS EMAIL_SENSITIVITY;

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.SENSITIVITY',
  'D402_TAG_LAB.SALES.CUSTOMERS.REGION', 'COLUMN'
) AS REGION_SENSITIVITY;
```

Expected: email is `RESTRICTED`; region inherits `CONFIDENTIAL`. A more specific label is an explicit choice, not a numeric ranking that Snowflake computes from words such as “restricted.”

Reference: [ALTER TABLE column operations](https://docs.snowflake.com/en/sql-reference/sql/alter-table-column).


## 12. Test inheritance and removing an override

A **direct assignment** is explicitly set on the object. An **effective value** is what applies after inheritance and overrides. Removing a direct assignment can reveal an inherited value rather than leave the object untagged. [Inheritance overrides](https://docs.snowflake.com/en/user-guide/object-tagging/inheritance#overriding-tag-inheritance).

```sql
ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS MODIFY COLUMN EMAIL
  UNSET TAG D402_TAG_LAB.GOVERNANCE.SENSITIVITY;

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.SENSITIVITY',
  'D402_TAG_LAB.SALES.CUSTOMERS.EMAIL', 'COLUMN'
) AS EMAIL_AFTER_UNSET;

-- Restore the reviewed column-specific classification.
ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS MODIFY COLUMN EMAIL
  SET TAG D402_TAG_LAB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';
```

Expected after the unset: `CONFIDENTIAL`, inherited from Sales. The final statement restores `RESTRICTED`.

The resulting hierarchy is:

```text
D402_TAG_LAB                         INTERNAL
└── SALES                            CONFIDENTIAL
    └── CUSTOMERS                    CONFIDENTIAL (inherited)
        ├── EMAIL                    RESTRICTED (direct)
        └── REGION                   CONFIDENTIAL (inherited)
```

In production, changing a tag associated with a policy may change protection. Review the effective result, not just whether an assignment was removed.

## 13. Create a view with view-column comments

A **view** is a named query over underlying data. Its columns are part of its own published interface and can have their own comments and tags.

This view renames `EMAIL` to `CONTACT_EMAIL`. We state column comments explicitly at creation. Do not assume source-column documentation becomes complete documentation for every derived output. [CREATE VIEW](https://docs.snowflake.com/en/sql-reference/sql/create-view).

```sql
CREATE VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V (
  CUSTOMER_ID COMMENT 'Synthetic reference copied from the customer source',
  CONTACT_EMAIL COMMENT 'Email alias for contact workflows; synthetic in this lab',
  REGION COMMENT 'Sales region copied from the customer source'
)
COMMENT = 'Contact-oriented projection; one row per synthetic customer'
AS
SELECT CUSTOMER_ID, EMAIL, REGION
FROM D402_TAG_LAB.SALES.CUSTOMERS;

DESCRIBE VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V;

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY',
  'D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V.CONTACT_EMAIL', 'COLUMN'
) AS CATEGORY_BEFORE_MANUAL_ASSIGNMENT;
```

Expected in this isolated lab: `NULL` for the category. We have not enabled propagation or directly tagged the view column. It can inherit Sales sensitivity, but the source table is not its containment parent.

## 14. Tag the view and its columns

Snowflake supports both view-level tags and view-column tags. The SQL command uses `ALTER VIEW`; the inspection functions below use domain `TABLE` for a view object and `COLUMN` for its column. [ALTER VIEW](https://docs.snowflake.com/en/sql-reference/sql/alter-view); [tag-inspection domains](https://docs.snowflake.com/en/sql-reference/functions/system_get_tag).

```sql
ALTER VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V SET TAG
  D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER = 'ANALYTICS',
  D402_TAG_LAB.GOVERNANCE.CONTAINS_PII = 'YES',
  D402_TAG_LAB.GOVERNANCE.RETENTION_CLASS = 'REPORTING_REVIEW';

ALTER VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V MODIFY COLUMN CONTACT_EMAIL
  SET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY = 'EMAIL',
          D402_TAG_LAB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';

ALTER VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V MODIFY COLUMN CUSTOMER_ID
  SET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY = 'CUSTOMER_ID';

ALTER VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V MODIFY COLUMN REGION
  SET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY = 'REGION';

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER',
  'D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V', 'TABLE'
) AS VIEW_OWNER_LABEL;
```

Expected owner label: `ANALYTICS`. The source table remains labeled `CUSTOMER_OPERATIONS`. Different teams can own responsibility for source data and a published interface.

A view retention label can mean “review whether this interface is still needed.” It does not independently delete the source records read by an ordinary view.

## 15. Maintain database, schema, table, and column comments

`COMMENT ON` replaces the object's comment. Keep descriptions useful: explain what the object means instead of merely repeating its name. Comments do not inherit like tags. [COMMENT](https://docs.snowflake.com/en/sql-reference/sql/comment).

```sql
COMMENT ON DATABASE D402_TAG_LAB
  IS 'Training only: reusable governance tags and synthetic customer assets';

COMMENT ON SCHEMA D402_TAG_LAB.SALES
  IS 'Synthetic customer source and contact interface; no production records';

COMMENT ON TABLE D402_TAG_LAB.SALES.CUSTOMERS
  IS 'One row per synthetic customer; illustrative counts, not a billing source';

COMMENT ON COLUMN D402_TAG_LAB.SALES.CUSTOMERS.EMAIL
  IS 'Synthetic contact address; classified as email for governance practice';

COMMENT ON VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V
  IS 'Contact interface over CUSTOMERS; excludes order-count information';

-- View columns support COMMENT ON COLUMN too.
COMMENT ON COLUMN D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V.CONTACT_EMAIL
  IS 'Alias of CUSTOMERS.EMAIL; preserve contact-data handling rules';

ALTER TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY
  SET COMMENT = 'Reviewed semantic category; maintained on source and view columns';
```

These statements document both the tag definition and the objects carrying tag assignments. Changing a comment does not change its associated tag values.

## 16. Inspect descriptions through metadata

**Information Schema** is a set of metadata views and functions describing database objects. Its `COLUMNS` view includes table and view columns, their order, types, and comments.

```sql
SHOW DATABASES LIKE 'D402_TAG_LAB';
SHOW SCHEMAS IN DATABASE D402_TAG_LAB;
SHOW TABLES IN SCHEMA D402_TAG_LAB.SALES;
SHOW VIEWS IN SCHEMA D402_TAG_LAB.SALES;
SHOW TAGS IN SCHEMA D402_TAG_LAB.GOVERNANCE;

SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE, COMMENT
FROM D402_TAG_LAB.INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'SALES'
ORDER BY TABLE_NAME, ORDINAL_POSITION;
```

Expected: seven column rows—four from the table and three from the view—with the descriptions from earlier steps. `SHOW` output includes comments for the relevant objects.

A **data dictionary** is a human-readable inventory of field meanings. A useful dictionary can combine these comments with tag metadata rather than store the same definitions in an unrelated spreadsheet.

Reference: [Information Schema COLUMNS](https://docs.snowflake.com/en/sql-reference/info-schema/columns).


## 17. Inspect object and column tag associations

`TAG_REFERENCES` returns tag associations for an object, including inherited associations. `TAG_REFERENCES_ALL_COLUMNS` returns associations across the columns of a table or view. Both respect the caller's visibility. [TAG_REFERENCES](https://docs.snowflake.com/en/sql-reference/functions/tag_references); [TAG_REFERENCES_ALL_COLUMNS](https://docs.snowflake.com/en/sql-reference/functions/tag_references_all_columns).

```sql
SELECT *
FROM TABLE(D402_TAG_LAB.INFORMATION_SCHEMA.TAG_REFERENCES(
  'D402_TAG_LAB.SALES.CUSTOMERS', 'TABLE'
));

SELECT *
FROM TABLE(D402_TAG_LAB.INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
  'D402_TAG_LAB.SALES.CUSTOMERS', 'TABLE'
));

SELECT *
FROM TABLE(D402_TAG_LAB.INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
  'D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V', 'TABLE'
));
```

Use `TABLE` as the second argument for the view too. Look at the tag name, value, column name, level, and assignment-method fields available in the results. They help explain where a value came from.

`SHOW TAGS` lists definitions. It does not replace association inspection. Also, a metadata query run without sufficient visibility cannot prove the absence of tags across the whole account.

## 18. Build a practical column catalog

This query joins column descriptions to effective tags. A **join** combines related records; **conditional aggregation** turns selected tag rows into readable output columns.

```sql
WITH COLUMN_TAGS AS (
  SELECT COLUMN_NAME, TAG_NAME, TAG_VALUE
  FROM TABLE(D402_TAG_LAB.INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'D402_TAG_LAB.SALES.CUSTOMERS', 'TABLE'
  ))
  WHERE TAG_DATABASE = 'D402_TAG_LAB'
    AND TAG_SCHEMA = 'GOVERNANCE'
)
SELECT
  C.COLUMN_NAME,
  C.DATA_TYPE,
  C.COMMENT,
  MAX(CASE WHEN T.TAG_NAME = 'DATA_CATEGORY' THEN T.TAG_VALUE END) AS DATA_CATEGORY,
  MAX(CASE WHEN T.TAG_NAME = 'SENSITIVITY' THEN T.TAG_VALUE END) AS SENSITIVITY,
  MAX(CASE WHEN T.TAG_NAME = 'BUSINESS_OWNER' THEN T.TAG_VALUE END) AS BUSINESS_OWNER
FROM D402_TAG_LAB.INFORMATION_SCHEMA.COLUMNS C
LEFT JOIN COLUMN_TAGS T ON C.COLUMN_NAME = T.COLUMN_NAME
WHERE C.TABLE_SCHEMA = 'SALES'
  AND C.TABLE_NAME = 'CUSTOMERS'
GROUP BY C.ORDINAL_POSITION, C.COLUMN_NAME, C.DATA_TYPE, C.COMMENT
ORDER BY C.ORDINAL_POSITION;
```

Expected: four rows. Email is restricted; the other fields inherit confidential sensitivity. All have a category and comment, and the owner is Customer Operations.

`MAX` is only pivoting the one effective value per tag in this single-value lab. It is not a rule for choosing the strongest sensitivity: alphabetical order is not a privacy ranking. A broader catalog should preserve object identifiers and deliberately handle any multi-value or conflict semantics.

## 19. Find missing metadata before publishing

A **metadata quality check** tests whether required documentation exists. Add a new unreviewed column to simulate **schema drift**, a change in table structure.

```sql
ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS ADD COLUMN SUPPORT_NOTE VARCHAR;

WITH CATEGORIES AS (
  SELECT DISTINCT COLUMN_NAME
  FROM TABLE(D402_TAG_LAB.INFORMATION_SCHEMA.TAG_REFERENCES_ALL_COLUMNS(
    'D402_TAG_LAB.SALES.CUSTOMERS', 'TABLE'
  ))
  WHERE TAG_DATABASE = 'D402_TAG_LAB'
    AND TAG_SCHEMA = 'GOVERNANCE'
    AND TAG_NAME = 'DATA_CATEGORY'
)
SELECT C.COLUMN_NAME,
       C.COMMENT IS NULL OR TRIM(C.COMMENT) = '' AS MISSING_COMMENT,
       T.COLUMN_NAME IS NULL AS MISSING_CATEGORY
FROM D402_TAG_LAB.INFORMATION_SCHEMA.COLUMNS C
LEFT JOIN CATEGORIES T ON C.COLUMN_NAME = T.COLUMN_NAME
WHERE C.TABLE_SCHEMA = 'SALES'
  AND C.TABLE_NAME = 'CUSTOMERS'
  AND (C.COMMENT IS NULL OR TRIM(C.COMMENT) = '' OR T.COLUMN_NAME IS NULL)
ORDER BY C.ORDINAL_POSITION;
```

Expected: `SUPPORT_NOTE` is flagged. It inherits broad sensitivity, but lacks a precise category and explanation. An inherited label is not proof of completed review.

A release process could stop publication when this query returns rows. Running the query alone does not implement that release gate. Free-text fields deserve particular review because they can contain unexpected personal details.

## 20. Extend the vocabulary and complete review

After reviewing the new field, extend the category list and document it. Adding an allowed value changes the vocabulary; assigning it classifies this specific column. [ALTER TAG](https://docs.snowflake.com/en/sql-reference/sql/alter-tag).

```sql
ALTER TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY
  ADD ALLOWED_VALUES 'FREE_TEXT';

ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS MODIFY COLUMN SUPPORT_NOTE
  SET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY = 'FREE_TEXT',
          D402_TAG_LAB.GOVERNANCE.SENSITIVITY = 'RESTRICTED';

COMMENT ON COLUMN D402_TAG_LAB.SALES.CUSTOMERS.SUPPORT_NOTE
  IS 'Synthetic-only support notes; free text requires personal-data review';

SELECT SYSTEM$GET_TAG_ALLOWED_VALUES(
  'D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY'
) AS UPDATED_CATEGORIES;
```

Rerun the section 19 check: expected zero rows. This shows completeness, not correctness of every label or effectiveness of protection.

Dropping an allowed value from a tag definition does not rewrite existing assignments that already use it. Migrate assignments before retiring vocabulary. [Allowed-value behavior](https://docs.snowflake.com/en/user-guide/object-tagging/work#set-a-list-of-allowed-tag-values).

## 21. Negative test: reject an unapproved value

A **negative test** checks that invalid input is rejected. Run this block separately because the error is intentional.

```sql
-- EXPECTED ERROR: TOP_SECRET is not in this tag's allowed-value list.
ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS MODIFY COLUMN EMAIL
  SET TAG D402_TAG_LAB.GOVERNANCE.SENSITIVITY = 'TOP_SECRET';
```

Expected: Snowflake rejects the assignment. The earlier `RESTRICTED` email value remains.

This protects consistency of the vocabulary. It does not prove that the person choosing an allowed value chose the correct classification.

## 22. Update and remove assignments

`SET TAG` updates an assignment. `UNSET TAG` removes that assignment, not the reusable tag definition. For view columns, use `ALTER VIEW ... MODIFY COLUMN` just as in the earlier assignment exercise. [ALTER VIEW tag operations](https://docs.snowflake.com/en/sql-reference/sql/alter-view).

```sql
ALTER VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V
  UNSET TAG D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER;

SELECT SYSTEM$GET_TAG(
  'D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER',
  'D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V', 'TABLE'
) AS VIEW_OWNER_AFTER_UNSET;

-- Restore the intentional owner of the published interface.
ALTER VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V
  SET TAG D402_TAG_LAB.GOVERNANCE.BUSINESS_OWNER = 'ANALYTICS';

ALTER VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V MODIFY COLUMN CONTACT_EMAIL
  UNSET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY;

ALTER VIEW D402_TAG_LAB.SALES.CUSTOMER_CONTACT_V MODIFY COLUMN CONTACT_EMAIL
  SET TAG D402_TAG_LAB.GOVERNANCE.DATA_CATEGORY = 'EMAIL';
```

Expected owner immediately after unset: `CUSTOMER_OPERATIONS`, inherited from Sales. The final state restores the view owner and category.

Other object-level removals use the same pattern with `ALTER DATABASE`, `ALTER SCHEMA`, or `ALTER TABLE` followed by `UNSET TAG` and the tag name. Never confuse removing a classification with deleting the underlying data.

## 23. Retire a tag definition safely

`DROP TAG` removes the reusable definition and affects its associations. Before removing a real tag, inspect assignments, policy connections, automated checks, and external catalog consumers. [DROP TAG](https://docs.snowflake.com/en/sql-reference/sql/drop-tag).

Use a disposable tag so the main exercise stays intact:

```sql
CREATE TAG D402_TAG_LAB.GOVERNANCE.REVIEW_NOTE
  COMMENT = 'Disposable tag for lifecycle practice';

ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS
  SET TAG D402_TAG_LAB.GOVERNANCE.REVIEW_NOTE = 'TRAINING_REVIEW_COMPLETE';

ALTER TABLE D402_TAG_LAB.SALES.CUSTOMERS
  UNSET TAG D402_TAG_LAB.GOVERNANCE.REVIEW_NOTE;

DROP TAG D402_TAG_LAB.GOVERNANCE.REVIEW_NOTE;

SHOW TAGS IN SCHEMA D402_TAG_LAB.GOVERNANCE;
```

Expected: the five main tags remain; the disposable tag is absent. We explicitly removed its lab assignment before retirement so the lifecycle is visible.

Removing a comment is a separate documentation action; it does not unset the object's tags.

## 24. Inheritance versus automatic propagation

**Propagation** carries tags through supported object dependencies or data movement. **Lineage** describes where data came from and how it was transformed. Neither is the same as containment inheritance.

| Mechanism | Example | What to verify |
|---|---|---|
| Inheritance | Sales schema classification reaches customer columns | Effective value and overrides |
| Manual assignment | Explicitly tag a view's renamed email column | Reviewed output meaning |
| Automatic propagation | A configured source tag reaches a supported derived object | Supported operation, configuration, conflicts, and timing |

Snowflake offers propagation modes for dependencies, data movement, or both. This is an Enterprise Edition capability and needs additional privileges. The core lab leaves propagation unconfigured so its results are predictable. [Automatic tag propagation](https://docs.snowflake.com/en/user-guide/object-tagging/propagation).

A transformation can change meaning: an email domain differs from a full email, but a rare domain may still reveal identity. A derived asset needs review even when metadata is propagated.

An external spreadsheet does not automatically receive Snowflake enforcement or metadata. Review exports and downstream catalogs separately.

## 25. From classification to enforcement

**Tag-based masking** associates a masking policy with a tag so compatible tagged columns receive protection. It requires Enterprise Edition or higher. A directly attached masking policy takes precedence over tag-based masking; review effective policy coverage. [Tag-based masking](https://docs.snowflake.com/en/user-guide/tag-based-masking-policies).

Our lab deliberately creates descriptive tags without attaching policies. Confirm the distinction:

```sql
SELECT CUSTOMER_ID, EMAIL, REGION
FROM D402_TAG_LAB.SALES.CUSTOMERS
ORDER BY CUSTOMER_ID;
```

Expected: original synthetic email values remain visible. `RESTRICTED` has not become a mask automatically.

A practical production process is:

1. Owner approves classification and use.
2. Steward applies controlled metadata.
3. Authorized administrator connects required policies.
4. Engineer tests results as intended consumer roles.
5. Monitoring checks new fields, overrides, and coverage gaps.

Applying a masking-enabled tag to a database or schema can affect many compatible child columns. Review that scope before making broad assignments.

## 26. Delegation and operating practices

A **data steward** maintains definitions and classifications. A **data owner** approves purpose and handling. A **tag administrator** controls vocabulary and tag privileges. These responsibilities can be split across roles.

For a hybrid model, a central team creates definitions and grants `APPLY` on selected tags to teams owning the target objects. Global `APPLY TAG` provides broader authority and should be deliberately assigned. The core lab needs no additional account-wide grant because its owner creates both definitions and targets. [Tag privilege approaches](https://docs.snowflake.com/en/user-guide/object-tagging/work#approaches-assigning-tagging-privileges).

Recommended review questions:

- Is the business owner a team with a clear contact process?
- Are required comments meaningful and current?
- Are inherited values being mistaken for reviewed column classifications?
- Does a new view have its own documented field meanings?
- Can a tag change alter a protection policy?
- Do catalog checks distinguish missing metadata from invisible metadata?
- Are lifecycle labels connected to actual workflows?

**Snowsight** is Snowflake's web interface. Use its object details and governance tag views to inspect descriptions and assignments when available. The SQL inspection blocks provide repeatable checks even if interface navigation changes.

## 27. Cleanup

Run after completing the lesson. These commands remove only the isolated lab database, its tags and objects, and its warehouse. Verify the names still belong to this exercise.

```sql
USE ROLE SYSADMIN;
USE SECONDARY ROLES NONE;

DROP DATABASE IF EXISTS D402_TAG_LAB;
DROP WAREHOUSE IF EXISTS D402_TAG_WH;

SHOW DATABASES LIKE 'D402_TAG_LAB';
SHOW WAREHOUSES LIKE 'D402_TAG_WH';
```

Expected: no active database or warehouse with these names. Database removal follows Snowflake recovery-retention behavior; it is not an immediate physical-erasure claim.

All tag definitions and assignments in this lab are confined to the lab database, so cleanup does not remove shared production vocabulary.

## 28. Practice questions and expected answers

1. **Does a database tag need to be defined inside that database?** No. A tag is a schema object and can be applied to other supported objects with appropriate privileges. This lab co-locates everything for easy cleanup.
2. **Does a business-owner tag transfer ownership?** No; it is descriptive metadata.
3. **Why is email restricted while region is confidential?** Email has a direct override; region inherits the schema value.
4. **Why did email remain classified after unsetting its tag?** The schema-level classification became effective.
5. **Can view columns have tags and comments?** Yes; the lab assigns both explicitly.
6. **Why did the new view column initially lack its source category?** Source dependency is different from containment inheritance; propagation was not configured.
7. **Does `SHOW TAGS` show every assignment?** No; use tag-reference functions to inspect associations.
8. **Does a completed metadata check prove security?** No; check actual roles and policies separately.
9. **What does removing an allowed value do to existing assignments?** It does not automatically rewrite them; migrate them deliberately.
10. **Why were emails still readable after tagging?** No masking policy was attached.

## 29. Abbreviations and official references

| Abbreviation | Expansion |
|---|---|
| SQL | Structured Query Language |
| PII | Personally Identifiable Information |
| ID | Identifier, as in customer identifier |
| WH | Warehouse, used in the lab object name |
| V | View, used as the view-name suffix |
| SYSADMIN | System Administrator, a Snowflake system role |

Official sources are linked beside the relevant concepts. For further reading, start with [object tagging](https://docs.snowflake.com/en/user-guide/object-tagging/introduction), [working with tags](https://docs.snowflake.com/en/user-guide/object-tagging/work), [inheritance](https://docs.snowflake.com/en/user-guide/object-tagging/inheritance), [comments](https://docs.snowflake.com/en/sql-reference/sql/comment), and [column tag inspection](https://docs.snowflake.com/en/sql-reference/functions/tag_references_all_columns).